## Exercise 3: AI Response Evaluation and Refinement Using Groq and Gemini

Implemented an AI response evaluation workflow using **Groq for response generation** and **Gemini for response evaluation**. The solution uses Pydantic for structured evaluation, evaluates generated responses for accuracy and relevance, and retries rejected responses using the evaluator's feedback. It also includes a patent-question test using Pig Latin to demonstrate the evaluation and retry mechanism.

In [1]:
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from groq import Groq
from google import genai
from pypdf import PdfReader
from pydantic import BaseModel

In [15]:
load_dotenv(override=True)

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is missing from your .env file")

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY is missing from your .env file")

groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

gemini_client = genai.Client(
    api_key=os.getenv("Google_API_KEY")
)

GROQ_MODEL = "openai/gpt-oss-20b"
GEMINI_MODEL = "gemini-3.5-flash-lite"

In [ ]:
def find_profile_file(filename):
    candidates = [
        Path("me") / filename,
        Path("twin") / filename,
        Path("../../twin") / filename,
    ]

    for path in candidates:
        if path.is_file():
            return path

    raise FileNotFoundError(
        f"Could not find {filename}. Checked: "
        + ", ".join(str(p.resolve()) for p in candidates)
    )


linkedin_path = find_profile_file("linkedin.pdf") 
summary_path = find_profile_file("summary.txt")

reader = PdfReader(str(linkedin_path))

linkedin = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

with open(summary_path, "r", encoding="utf-8") as f:
    summary = f.read()

print("LinkedIn PDF:", linkedin_path.resolve())
print("Summary file:", summary_path.resolve())
print("LinkedIn characters:", len(linkedin))
print("Summary characters:", len(summary))

In [6]:
name = "Hiren Trivedi"

system_prompt = f"""
You are acting as {name}.

You are answering questions on {name}'s website, particularly
questions about {name}'s career, background, skills, and experience.

Represent {name} faithfully using the provided summary and LinkedIn profile.

Be professional, friendly, and engaging, as if speaking with a potential
client or future employer.

Do not invent qualifications, experience, achievements, or personal details.
If you do not know the answer from the provided information, say so.

## Summary
{summary}

## LinkedIn Profile
{linkedin}

Stay in character as {name}.
"""

In [7]:
def normalize_history(history):
    messages = []

    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        elif isinstance(item, (tuple, list)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({
                    "role": "user",
                    "content": str(user_message)
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": str(assistant_message)
                })

    return messages


def chat(message, history):
    messages = [
        {"role": "system", "content": system_prompt}
    ]

    messages.extend(normalize_history(history))
    messages.append({
        "role": "user",
        "content": message
    })

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages
    )

    return response.choices[0].message.content or ""

In [ ]:
print(chat("Tell me about your professional background.", []))

In [9]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [10]:
evaluator_system_prompt = f"""
You are an evaluator deciding whether an AI-generated response is acceptable.

The agent is representing {name} on a professional website.
The agent must be professional, engaging, accurate, and faithful to the
provided profile.

Reject answers that contain fabricated facts, materially inaccurate
claims, irrelevant content, or responses that fail to follow the
required professional behavior.

Evaluate the answer using the available conversation and profile.
Do not reject a response merely because it admits that information
is unavailable.

## Summary
{summary}

## LinkedIn Profile
{linkedin}

Return a structured evaluation containing:
- is_acceptable: true if the answer meets the requirements, otherwise false
- feedback: a concise explanation of your decision
"""

In [11]:
def evaluator_user_prompt(reply, message, history):
    return f"""
Conversation history:
{normalize_history(history)}

Latest user message:
{message}

Latest agent response:
{reply}

Evaluate whether the latest agent response is acceptable.
Return your decision and feedback.
"""

In [12]:
def evaluate(reply, message, history) -> Evaluation:
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=evaluator_user_prompt(reply, message, history),
        config={
            "system_instruction": evaluator_system_prompt,
            "response_mime_type": "application/json",
            "response_schema": Evaluation,
            "temperature": 0
        }
    )

    if response.parsed is not None:
        return Evaluation.model_validate(response.parsed)

    if response.text:
        return Evaluation.model_validate_json(response.text)

    raise ValueError("Gemini returned an empty evaluation.")

In [13]:
def evaluate(reply, message, history) -> Evaluation:
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=evaluator_user_prompt(reply, message, history),
        config={
            "system_instruction": evaluator_system_prompt,
            "response_mime_type": "application/json",
            "response_schema": Evaluation,
            "temperature": 0
        }
    )

    if response.parsed is not None:
        return Evaluation.model_validate(response.parsed)

    if response.text:
        return Evaluation.model_validate_json(response.text)

    raise ValueError("Gemini returned an empty evaluation.")

In [ ]:
test_message = "Tell me about your professional background."
test_reply = chat(test_message, [])

evaluation = evaluate(test_reply, test_message, [])

print(evaluation)

In [17]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"""

## Previous answer rejected

Your previous response did not pass quality control.

## Your attempted answer
{reply}

## Reason for rejection
{feedback}

Produce a corrected answer that addresses the feedback.
Use only the facts supported by the provided profile.
"""

    messages = [
        {"role": "system", "content": updated_system_prompt}
    ]

    messages.extend(normalize_history(history))
    messages.append({
        "role": "user",
        "content": message
    })

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages
    )

    return response.choices[0].message.content or ""

In [ ]:
test_message = "Tell me about your professional background."
first_reply = chat(test_message, [])
test_evaluation = evaluate(first_reply, test_message, [])

if test_evaluation.is_acceptable:
    final_reply = first_reply
else:
    final_reply = rerun(
        first_reply,
        test_message,
        [],
        test_evaluation.feedback
    )

print(final_reply)

In [19]:
def chat(message, history):
    history = normalize_history(history)

    if "patent" in message.lower():
        current_system_prompt = system_prompt + """
For this test only, respond entirely in Pig Latin.
"""
    else:
        current_system_prompt = system_prompt

    messages = [
        {"role": "system", "content": current_system_prompt}
    ]

    messages.extend(history)
    messages.append({
        "role": "user",
        "content": message
    })

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages
    )

    reply = response.choices[0].message.content or ""

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print("Feedback:", evaluation.feedback)

        reply = rerun(
            reply,
            message,
            history,
            evaluation.feedback
        )

    return reply

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    title="My AI Digital Twin",
    description="Ask questions about professional background, skills, projects, and experience."
)

demo.launch()